# Hands-on Modul 3.4: Simulasi Pipeline LLMOps (Versioning to Deployment) 🏭

Tantangan terbesar AI Engineering bukan melatih model, tapi **memeliharanya**.
Di sini kita akan mempraktikkan siklus hidup model:
**Train -> Track -> Register -> Containerize**.

In [1]:
# Instalasi MLflow untuk tracking
!pip install mlflow transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.3 MB/s eta 0:00:00


In [2]:
import mlflow
import os
import shutil
from random import random

# Bersihkan run lama agar bersih (opsional)
if os.path.exists("mlruns"):
    try:
        shutil.rmtree("mlruns")
    except:
        pass

# 1. Setup MLflow (Lokal)
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Llama3-Production-Lifecycle")

# --- FIX: Membuat Kelas Model Dummy ---
# Agar bisa diregister, kita butuh objek yang dianggap "Model" oleh MLflow
class DummyModel(mlflow.pyfunc.PythonModel):
    def predict(self, context, model_input):
        return "Ini hanya simulasi model"

print("Memulai Eksperimen Simulasi...")

# 2. Simulasi Training Loop
with mlflow.start_run(run_name="FineTuning-v1-LoRA") as run:

    # A. Log Parameters
    params = {
        "learning_rate": 2e-4,
        "batch_size": 4,
        "lora_rank": 16,
        "dataset": "alpaca-cleaned"
    }
    mlflow.log_params(params)

    # B. Log Metrics
    for epoch in range(1, 6):
        simulated_loss = 2.0 / epoch + (random() * 0.1)
        mlflow.log_metric("train_loss", simulated_loss, step=epoch)
        print(f"Epoch {epoch}: Loss {simulated_loss:.4f}")

    # C. Log Model (YANG SUDAH DIPERBAIKI)
    # Kita gunakan .log_model(), bukan .log_artifacts()
    # Ini otomatis membuat file metadata 'MLmodel' yang wajib ada untuk registrasi
    print("Menyimpan model ke MLflow...")
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=DummyModel()
    )

    # D. Register Model
    # Sekarang ini akan berhasil karena 'model' sudah punya metadata yang benar
    model_uri = f"runs:/{run.info.run_id}/model"
    mlflow.register_model(model_uri, "Llama3-Chatbot-Prod")

print(f"\n✅ Sukses! Model tersimpan dengan Run ID: {run.info.run_id}")
print("Cek folder 'mlruns' di panel kiri untuk melihat database tracking.")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/01 11:43:13 INFO mlflow.tracking.fluent: Experiment with name 'Llama3-Production-Lifecycle' does not exist. Creating a new experiment.
/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Memulai Eksperimen Simulasi...


2026/04/01 11:43:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 11:43:14 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Epoch 1: Loss 2.0154
Epoch 2: Loss 1.0777
Epoch 3: Loss 0.6681
Epoch 4: Loss 0.5429
Epoch 5: Loss 0.4465
Menyimpan model ke MLflow...


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'Llama3-Chatbot-Prod'.
2026/04/01 11:43:21 WARNING mlflow.tracking._model_registry.fluent: Run with id db840eba22eb4a6fbe50bfa63ada0161 has no artifacts at artifact path 'model', registering model based on models:/m-742eae01b04e414a893752c5d9ffcd4b instead



✅ Sukses! Model tersimpan dengan Run ID: db840eba22eb4a6fbe50bfa63ada0161
Cek folder 'mlruns' di panel kiri untuk melihat database tracking.


Created version '1' of model 'Llama3-Chatbot-Prod'.


## Langkah 2: Infrastructure as Code (IaC) - Membuat Dockerfile

Sekarang kita punya model yang terdaftar. Bagaimana cara mengirimnya ke server? Kita harus membungkusnya.
Jalankan sel di bawah untuk **men-generate file Dockerfile** secara otomatis.

In [3]:
# Kita menulis string konten ke file fisik 'Dockerfile'

dockerfile_content = """
# 1. Base Image: Wajib pakai versi GPU/CUDA
FROM nvidia/cuda:12.1.0-runtime-ubuntu22.04

# 2. Setup Environment
ENV DEBIAN_FRONTEND=noninteractive
WORKDIR /app

# 3. Install System Deps
RUN apt-get update && apt-get install -y \\
    python3 python3-pip git \\
    && rm -rf /var/lib/apt/lists/*

# 4. Install Python Libs
# Trik: Install torch dulu (berat), baru yang lain
RUN pip3 install --no-cache-dir torch torchvision torchaudio
COPY requirements.txt .
RUN pip3 install --no-cache-dir -r requirements.txt

# 5. Copy Application Code
COPY . .

# 6. Command Start
# Menjalankan server API (misal pakai vLLM atau FastAPI)
CMD ["python3", "app.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("✅ File 'Dockerfile' berhasil dibuat!")
print("Isi file:")
print("-" * 20)
print(dockerfile_content)

✅ File 'Dockerfile' berhasil dibuat!
Isi file:
--------------------

# 1. Base Image: Wajib pakai versi GPU/CUDA
FROM nvidia/cuda:12.1.0-runtime-ubuntu22.04

# 2. Setup Environment
ENV DEBIAN_FRONTEND=noninteractive
WORKDIR /app

# 3. Install System Deps
RUN apt-get update && apt-get install -y \
    python3 python3-pip git \
    && rm -rf /var/lib/apt/lists/*

# 4. Install Python Libs
# Trik: Install torch dulu (berat), baru yang lain
RUN pip3 install --no-cache-dir torch torchvision torchaudio
COPY requirements.txt .
RUN pip3 install --no-cache-dir -r requirements.txt

# 5. Copy Application Code
COPY . .

# 6. Command Start
# Menjalankan server API (misal pakai vLLM atau FastAPI)
CMD ["python3", "app.py"]



## Langkah 3: Orkestrasi - Membuat Kubernetes Manifest

Untuk skala besar, kita butuh Kubernetes.
Jalankan sel ini untuk membuat file `deployment.yaml` yang mendefinisikan kebutuhan GPU.

In [4]:
k8s_content = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: llama3-inference
spec:
  replicas: 2 # Load balancing 2 server
  selector:
    matchLabels:
      app: llama3
  template:
    metadata:
      labels:
        app: llama3
    spec:
      # --- KUNCI: GPU Scheduling ---
      # Pastikan Pod ini hanya mendarat di Node yang punya GPU
      tolerations:
      - key: "sku"
        operator: "Equal"
        value: "gpu-a100"
        effect: "NoSchedule"

      containers:
      - name: llama3-container
        image: my-registry/llama3:v1
        ports:
        - containerPort: 8000

        # --- KUNCI: Request GPU ---
        resources:
          limits:
            nvidia.com/gpu: 1 # Minta 1 GPU penuh
"""

with open("deployment.yaml", "w") as f:
    f.write(k8s_content)

print("✅ File 'deployment.yaml' berhasil dibuat!")

✅ File 'deployment.yaml' berhasil dibuat!


### Kesimpulan Hands-on

Anda telah mensimulasikan tugas seorang **MLOps Engineer**:
1.  Melacak eksperimen agar tidak hilang (**MLflow**).
2.  Menyiapkan resep instalasi otomatis (**Dockerfile**).
3.  Menyiapkan instruksi deployment skala besar (**Kubernetes**).

File `Dockerfile` dan `deployment.yaml` yang baru saja Anda buat adalah artefak nyata yang bisa Anda serahkan ke tim IT untuk men-deploy model Anda ke server produksi mana pun di dunia.